# 🧮 Naive Bayes
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> Naive Bayes extends the Bayesian classification idea with one strong simplification: assume features are conditionally independent given the class. Despite this "naive" assumption often being wrong, it works surprisingly well in practice — especially for text.

### What you'll learn
- The conditional independence assumption and why it's "naive"
- Gaussian Naive Bayes for numeric features
- Multinomial Naive Bayes for text classification
- Why NB works well even when the independence assumption is violated
- NB vs logistic regression — when to use each

### Dataset: SMS Spam (text classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

## 📐 Part 1 — Bayes with Independence Assumption

Full Bayes would require estimating P(X₁=x₁, X₂=x₂, ..., Xₚ=xₚ | Y=k) — the joint distribution of all features. With p=1000 features this is impossible.

**Naive Bayes** assumes conditional independence:
```
P(X₁,...,Xₚ | Y=k) = P(X₁|Y=k) × P(X₂|Y=k) × ... × P(Xₚ|Y=k)
```

Now we only need to estimate p separate univariate distributions — completely tractable.

The "naive" part: features are rarely truly independent. But the resulting classifier is surprisingly robust — the probability estimates may be wrong, but the *decision* (which class gets the highest probability) is often right.

In [ ]:
# Wine Quality dataset — predict whether a wine is high quality (score >= 7)
# Real sensory + chemical measurements from Portuguese wines
# Much more meaningful than Iris for business analytics students

import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.stats import norm
import matplotlib.pyplot as plt

try:
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    wine = pd.read_csv(url, sep=';')
    print("Loaded Wine Quality dataset from UCI")
except:
    from sklearn.datasets import load_wine
    data = load_wine(as_frame=True)
    wine = data.frame.rename(columns={'target': 'quality'})
    wine['quality'] = wine['quality'] * 3  # scale to 0-12 range
    print("Using sklearn wine dataset (fallback)")

# Binary target: quality >= 7 = "high quality"
wine['high_quality'] = (wine['quality'] >= 7).astype(int)
feature_cols = [c for c in wine.columns if c not in ['quality','high_quality']]
X_wine = wine[feature_cols]
y_wine = wine['high_quality']

print(f"Wine dataset: {wine.shape}")
print(f"High quality (score >= 7): {y_wine.mean():.1%}")
print(f"Features: {feature_cols}")
wine.head(3)


In [ ]:
# Gaussian NB on Wine Quality — show what it learns
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.stats import norm

X_wine_arr = X_wine.values
X_tr, X_te, y_tr, y_te = train_test_split(X_wine_arr, y_wine,
                                            test_size=0.25, random_state=42,
                                            stratify=y_wine)
gnb = GaussianNB()
gnb.fit(X_tr, y_tr)

print("=== Gaussian NB: Wine Quality ===")
print(classification_report(y_te, gnb.predict(X_te),
                            target_names=['Standard','High Quality']))

# Visualize: what distributions GNB learned for top features
top_features = ['alcohol', 'volatile acidity', 'sulphates']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, feat in zip(axes, top_features):
    feat_idx = list(X_wine.columns).index(feat)
    for cls, color, label in [(0,'#1e5fa8','Standard (<=6)'),
                               (1,'#e85d20','High Quality (>=7)')]:
        mu  = gnb.theta_[cls, feat_idx]
        std = np.sqrt(gnb.var_[cls, feat_idx])
        x_range = np.linspace(mu - 3.5*std, mu + 3.5*std, 200)
        ax.plot(x_range, norm.pdf(x_range, mu, std), color=color, lw=2, label=label)
        data_cls = X_wine.values[y_wine==cls, feat_idx]
        ax.hist(data_cls, bins=20, color=color, alpha=0.2, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle("Gaussian NB: Learned distributions per class\n(separation = predictive power)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 Alcohol content: high-quality wines cluster at higher values")
print("   Volatile acidity: high-quality wines have LOWER acidity (less vinegary)")
print("   Where the distributions overlap least = most discriminating feature")


In [ ]:
# Text classification: spam detection
X_text = sms['message'].values
y_text = sms['spam'].values

# Pipeline: TF-IDF → Multinomial NB
pipeline_mnb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf',   MultinomialNB(alpha=1.0))  # alpha = Laplace smoothing
])

X_tr, X_te, y_tr, y_te = train_test_split(X_text, y_text, test_size=0.2, 
                                             random_state=42, stratify=y_text)
pipeline_mnb.fit(X_tr, y_tr)
y_pred = pipeline_mnb.predict(X_te)

print("=== Multinomial Naive Bayes: Spam Detection ===\n")
print(classification_report(y_te, y_pred, target_names=['Ham','Spam']))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Ham','Spam']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Spam Detection')
plt.tight_layout()
plt.show()

In [ ]:
# Most discriminative words for spam vs ham
import re
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_vec = vectorizer.fit_transform(X_text)
mnb = MultinomialNB().fit(X_vec, y_text)

feature_names = vectorizer.get_feature_names_out()
# Log-ratio: how much more likely in spam vs ham
log_ratios = mnb.feature_log_prob_[1] - mnb.feature_log_prob_[0]

top_spam = pd.Series(log_ratios, index=feature_names).nlargest(15)
top_ham  = pd.Series(log_ratios, index=feature_names).nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(top_spam.index[::-1], top_spam.values[::-1], color='#e85d20', edgecolor='white')
axes[0].set_title('Top 15 Spam Words')
axes[0].set_xlabel('Log-ratio (spam/ham)')

axes[1].barh(top_ham.index, top_ham.abs().values, color='#1e5fa8', edgecolor='white')
axes[1].set_title('Top 15 Ham Words')
axes[1].set_xlabel('Log-ratio (ham/spam)')

plt.suptitle('Most Discriminative Words — Naive Bayes Spam Classifier', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 The model learned these patterns purely from labeled examples — no rules were written by hand")

In [ ]:
# @title 📝 Quiz — Naive Bayes
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the conditional independence assumption in Naive Bayes?
# @markdown **Q2:** What type of Naive Bayes would you use for word counts in documents?
# @markdown **Q3:** What is Laplace smoothing and why is it needed?
# @markdown **Q4:** Why does Naive Bayes sometimes outperform logistic regression on small datasets?
# @markdown **Q5:** What is one real-world scenario where the independence assumption is clearly violated but NB still works?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Naive Bayes"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*